# 山海行记 · uni-app 多端项目对照学习笔记

> 配套项目：`D:\web\uniapp-app`
> 在线预览：<https://bell-567.github.io/web/uniapp/>
> 技术栈：uni-app（Vue 3 + Vite）· HBuilderX 5.34

## 怎么用这个 notebook

1. 每一节先读说明，再对照下面的代码。
2. 代码块**只用于阅读，不能直接运行**——uni API 需要 HBuilderX / 微信开发者工具 / 手机基座。
3. 想动手时，修改 `D:\web\uniapp-app\src` 下的源文件，然后运行 `npm run dev:h5`。
4. 为节省篇幅，页面与组件的 `<style scoped>` 已省略，样式请打开源文件查看；样式体系在第 7 章单独讲。

## 推荐阅读顺序

1. `pages.json`（先知道有哪几个页面）
2. 首页 `pages/index/index.vue`（看页面是怎么组装起来的）
3. `AppShell.vue`（所有页面共用的外壳）
4. `composables/`（共享状态与逻辑）
5. 列表页 → 详情页 → 行程页 → 关于页
6. 最后按分类看 25 个组件

## 目录

- 第 0 章　一张图看懂运行链路
- 第 1 章　工程配置（package.json / manifest.json / pages.json）
- 第 2 章　启动入口（main.js / App.vue）
- 第 3 章　组合式函数 composables
- 第 4 章　数据层 destinations.js
- 第 5 章　五个页面
- 第 6 章　二十五个组件
- 第 7 章　样式体系与多端兼容
- 第 8 章　三条完整数据流
- 第 9 章　Web → uni-app 迁移对照
- 第 10 章　自测题与动手练习
- 附录　运行命令与排错


## 第 0 章　一张图看懂运行链路

```text
index.html (仅 H5 需要)
  └─ src/main.js            createSSRApp(App)，导出 createApp 工厂
       └─ src/App.vue       全局样式 + 应用级生命周期（onLaunch/onShow/onHide）
            └─ src/pages.json   页面路由表；数组第一项就是启动页
                 └─ src/pages/xxx.vue   具体页面
                      ├─ src/components/*.vue   页面用到的组件
                      ├─ src/composables/*.js   跨页面共享的状态与逻辑
                      └─ src/data/destinations.js  模拟数据
```

可以把它分成三层：

| 层 | 文件 | 职责 |
| --- | --- | --- |
| 配置层 | `package.json`、`manifest.json`、`pages.json` | 告诉编译器「有哪些端、哪些页面、每端怎么配置」 |
| 运行时层 | `main.js`、`App.vue` | 创建应用、注册全局样式与生命周期 |
| 业务层 | `pages/`、`components/`、`composables/`、`data/` | 页面、组件、共享状态、数据 |

> 关键区别：普通 Vue 项目由 `main.js` 手动 `mount('#app')`；uni-app 由各端运行时调用
> `createApp()` 工厂，再自行挂载。所以 `main.js` 里看不到 `mount`。

### 🧠 先记住三组 API

| 场景 | Web 版 | uni-app 版 |
| --- | --- | --- |
| 页面跳转 | `router.push` | `uni.navigateTo` / `uni.switchTab` |
| 本地存储 | `localStorage` | `uni.getStorageSync` / `uni.setStorageSync` |
| 页面滚动 | `window.scrollY` / `window.scrollTo` | `onPageScroll` / `uni.pageScrollTo` |


## 第 1 章　工程配置

这一章的文件决定了「项目怎么编译、编译到哪些端、有哪些页面」。一般不需要天天改，
但面试/答辩时最容易被问「uni-app 的路由在哪配置」。

### 1.1 package.json

**作用**：声明依赖和 npm 脚本。

- `scripts` 里 `dev:*` 是开发模式（监听改动），`build:*` 是生产构建，默认端是 H5。
- `@dcloudio/uni-h5`、`@dcloudio/uni-mp-weixin`、`@dcloudio/uni-app-plus` 分别是三端的运行时。
- `@dcloudio/vite-plugin-uni` 是编译核心，`vite` 版本被锁定为 5.2.8，避免与 uni 插件不兼容。
- 版本号 `3.0.0-5030420261029002` 与 HBuilderX 5.34 配套，所有 `@dcloudio/*` 必须保持一致。


```json
{
  "name": "wanderlog-uniapp",
  "version": "1.0.0",
  "private": true,
  "description": "山海行记 · 环球旅行手账（uni-app 多端版本）",
  "scripts": {
    "dev:h5": "uni",
    "dev:mp-weixin": "uni -p mp-weixin",
    "dev:app": "uni -p app",
    "build:h5": "uni build",
    "build:mp-weixin": "uni build -p mp-weixin",
    "build:app": "uni build -p app"
  },
  "dependencies": {
    "@dcloudio/uni-app": "3.0.0-5030420261029002",
    "@dcloudio/uni-app-plus": "3.0.0-5030420261029002",
    "@dcloudio/uni-components": "3.0.0-5030420261029002",
    "@dcloudio/uni-h5": "3.0.0-5030420261029002",
    "@dcloudio/uni-mp-weixin": "3.0.0-5030420261029002",
    "vue": "^3.4.21"
  },
  "devDependencies": {
    "@dcloudio/types": "^3.4.8",
    "@dcloudio/uni-cli-shared": "3.0.0-5030420261029002",
    "@dcloudio/vite-plugin-uni": "3.0.0-5030420261029002",
    "@vue/runtime-core": "^3.4.21",
    "rollup": "4.14.3",
    "vite": "5.2.8"
  }
}
```

### 1.2 vite.config.js

**作用**：注册 uni 插件。普通 Vite 项目还需要 Vue 插件、别名等配置；
uni-app 的 `uni()` 插件已经内置了 Vue 支持与多端编译，所以这里只有一行。


```js
import { defineConfig } from 'vite'
import uni from '@dcloudio/vite-plugin-uni'

// uni-app Vue3 + Vite 工程：H5 / 小程序 / App 共用同一套源码
export default defineConfig({
  plugins: [uni()],
})
```

### 1.3 index.html

**作用**：**H5 专用**的 HTML 模板。小程序和 App 不需要它。

- 用一小段脚本动态写入 `viewport`，支持刘海屏安全区（`viewport-fit=cover`）。
- `<div id="app">` 是挂载点；`<!--app-html-->` 是 uni-app 编译时的占位注释。
- 小程序端不会读取这个文件，页面结构由 `pages.json` + 页面组件决定。

⚠️ 容易误解：改这里的 `<title>` 只影响 H5 首屏标题；各页面导航栏标题在 `pages.json` 里配置。


```html
<!DOCTYPE html>
<html lang="zh-CN">
  <head>
    <meta charset="UTF-8" />
    <script>
      var coverSupport =
        'CSS' in window &&
        typeof CSS.supports === 'function' &&
        (CSS.supports('top: env(a)') || CSS.supports('top: constant(a)'))
      document.write(
        '<meta name="viewport" content="width=device-width, user-scalable=no, initial-scale=1.0, maximum-scale=1.0, minimum-scale=1.0' +
          (coverSupport ? ', viewport-fit=cover' : '') +
          '" />'
      )
    </script>
    <title>山海行记 · 环球旅行手账</title>
    <!--preload-links-->
    <!--app-context-->
  </head>
  <body>
    <div id="app"><!--app-html--></div>
    <script type="module" src="/src/main.js"></script>
  </body>
</html>
```

### 1.4 src/manifest.json

**作用**：**各端专属配置**，相当于「一套源码，多端身份证」。

| 字段 | 说明 |
| --- | --- |
| `name` / `versionName` / `versionCode` | 应用名称与版本（App 打包用） |
| `appid` | DCloud 应用标识；当前为空，真机运行/云打包前需要登录 HBuilderX 获取 |
| `app-plus` | App 端：启动图、Android 权限（联网、网络状态） |
| `mp-weixin` | 微信小程序：测试号 `touristappid`、关闭域名校验（仅开发） |
| `h5.router` | H5 路由：`hash` 模式 + `base` 子路径 |
| `vueVersion` | 固定为 `"3"`，表示使用 Vue 3 编译器 |

⚠️ `urlCheck: false` 只是开发期跳过域名校验，**正式发布**时图片域名仍需在小程序后台配置，
或者把图片下载到 `src/static/` 使用本地路径。


```json
{
  "name": "山海行记",
  "appid": "",
  "description": "环球旅行手账：目的地图鉴、行程规划、收藏与留言",
  "versionName": "1.0.0",
  "versionCode": "100",
  "transformPx": false,
  "app-plus": {
    "usingComponents": true,
    "nvueStyleCompiler": "uni-app",
    "compilerVersion": 3,
    "splashscreen": {
      "alwaysShowBeforeRender": true,
      "waiting": true,
      "autoclose": true,
      "delay": 0
    },
    "modules": {},
    "distribute": {
      "android": {
        "permissions": [
          "<uses-permission android:name=\"android.permission.INTERNET\"/>",
          "<uses-permission android:name=\"android.permission.ACCESS_NETWORK_STATE\"/>"
        ]
      },
      "ios": {},
      "sdkConfigs": {}
    }
  },
  "quickapp": {},
  "mp-weixin": {
    "appid": "touristappid",
    "setting": {
      "urlCheck": false,
      "es6": true,
      "postcss": true,
      "minified": true
    },
    "usingComponents": true
  },
  "h5": {
    "title": "山海行记 · 环球旅行手账",
    "router": {
      "mode": "hash",
      "base": "/"
    }
  },
  "uniStatistics": {
    "enable": false
  },
  "vueVersion": "3"
}
```

### 1.5 src/pages.json

**作用**：页面路由表 + 原生导航栏 + 原生 tabBar。这是 uni-app 最重要的一份配置。

- `pages` 数组的顺序就是路由注册顺序，**第一项是启动页**。
- 每一项的 `style.navigationBarTitleText` 是该页面顶部导航栏标题。
- `globalStyle` 设置全局导航栏颜色、背景色。
- `tabBar.list` 里的 4 个页面会生成底部 tab；**只有这 4 个页面能用 `switchTab` 跳转**。
- 详情页 `pages/destination/detail` 不在 tabBar 中，所以必须用 `navigateTo`，并且可以带 query。

```text
tabBar 4 个：首页 / 目的地 / 行程规划 / 关于
非 tabBar 1 个：目的地详情（从列表页 navigateTo 进入）
```

💡 新增一个 tab 页面时，要同时改 `pages` 数组和 `tabBar.list`；
tabBar 最多 5 项，且每项至少要有 `pagePath` 和 `text`（图标可选）。


```json
{
  "pages": [
    {
      "path": "pages/index/index",
      "style": {
        "navigationBarTitleText": "山海行记"
      }
    },
    {
      "path": "pages/destinations/index",
      "style": {
        "navigationBarTitleText": "探索目的地"
      }
    },
    {
      "path": "pages/destination/detail",
      "style": {
        "navigationBarTitleText": "目的地详情"
      }
    },
    {
      "path": "pages/planner/index",
      "style": {
        "navigationBarTitleText": "行程规划"
      }
    },
    {
      "path": "pages/about/index",
      "style": {
        "navigationBarTitleText": "关于山海行记"
      }
    }
  ],
  "globalStyle": {
    "navigationBarTextStyle": "black",
    "navigationBarTitleText": "山海行记",
    "navigationBarBackgroundColor": "#F7F3EC",
    "backgroundColor": "#F7F3EC",
    "backgroundTextStyle": "dark"
  },
  "tabBar": {
    "color": "#776C5E",
    "selectedColor": "#D96C4B",
    "backgroundColor": "#FFFFFF",
    "borderStyle": "black",
    "list": [
      {
        "pagePath": "pages/index/index",
        "text": "首页"
      },
      {
        "pagePath": "pages/destinations/index",
        "text": "目的地"
      },
      {
        "pagePath": "pages/planner/index",
        "text": "行程规划"
      },
      {
        "pagePath": "pages/about/index",
        "text": "关于"
      }
    ]
  }
}
```

### 1.6 src/uni.scss

**作用**：uni-app 约定俗成的 SCSS 变量文件，主要给插件市场的组件用。

本项目全部使用普通 CSS，没有安装 `sass`，所以这些变量**不会生效**，
保留它只是为了兼容第三方 uni-ui 组件。


```scss
/**
 * uni-app 内置样式变量（普通 CSS 项目可不使用；保留以兼容插件市场组件）
 * 如需 SCSS，请先安装 sass，并在 <style lang="scss"> 中使用。
 */
$uni-color-primary: #d96c4b;
$uni-color-success: #2f9e6e;
$uni-color-warning: #e9c46a;
$uni-color-error: #c0392b;
```

## 第 2 章　启动入口

### 2.1 src/main.js

**作用**：创建 Vue 应用实例。

三端共用同一份入口，但挂载方式不同，所以 uni-app 规定必须：

1. 使用 `createSSRApp`（而不是 `createApp`）；
2. 导出一个名为 `createApp` 的**工厂函数**；
3. 返回 `{ app }`。

`createSSRApp` 让同一份代码既能在浏览器做客户端渲染，也能在小程序/App 的运行时里工作。


```js
import { createSSRApp } from 'vue'
import App from './App.vue'

// uni-app 要求使用 createSSRApp 并导出 createApp 工厂函数
export function createApp() {
  const app = createSSRApp(App)
  return { app }
}
```

### 2.2 src/App.vue

**作用**：应用级生命周期 + **全局样式**。

#### 应用生命周期

- `onLaunch`：小程序/App 启动时执行一次。
- `onShow`：应用从后台切回前台。
- `onHide`：应用切到后台。

注意这是 **App 级**生命周期，不是页面级。页面级的是 `onLoad`、`onShow`、`onReady` 等。

#### 全局样式

1. `page` 选择器设置整页背景、字体、行高（小程序没有 `body`，用 `page`）。
2. 重置 `button` 的默认边框、内边距（小程序按钮默认样式很重）。
3. `.app` 定义一整套 CSS 变量：颜色、阴影、圆角、最大宽度、标题字体。
4. `.app.theme-dark` 只覆盖变量值，就实现整站深色主题。
5. `.container`、`.section`、`.btn`、`.card`、`.chip`、`.tag`、`.field`、`.page-head`
   是全站复用的公共类。

#### 为什么主题变量不写在 `page` 上？

因为小程序里 `page` 是运行时节点，不能给它动态加 class。主题 class 必须加在每个页面
最外层的 `.app` 上（由 `AppShell` 负责），切换 class 后，所有 `var(--xxx)` 自动变色。

#### 为什么 App.vue 不能像 Vue 网站那样包住所有页面？

uni-app 的每个页面是独立编译的，`App.vue` 只提供全局样式和生命周期，不能当布局容器。
所以项目写了 `AppShell.vue`，在每个页面里复用「导航栏 + 内容 + 页脚」。


```vue
<script>
export default {
  onLaunch() {
    console.log('山海行记 App Launch')
  },
  onShow() {
    console.log('山海行记 App Show')
  },
  onHide() {
    console.log('山海行记 App Hide')
  },
}
</script>

<style>
/* ============================================================
   全局样式（uni-app 中 App.vue 的样式作用于所有页面）
   ============================================================ */
page {
  background-color: #f7f3ec;
  color: #2a241d;
  font-family: -apple-system, 'PingFang SC', 'Microsoft YaHei', 'Helvetica Neue', sans-serif;
  line-height: 1.7;
  -webkit-font-smoothing: antialiased;
}

view,
text,
button,
input,
textarea,
image {
  box-sizing: border-box;
}

text {
  color: inherit;
}

image {
  display: block;
}

/* 重置小程序 button 的默认样式 */
button {
  margin: 0;
  padding: 0;
  background: transparent;
  border: none;
  border-radius: 0;
  color: inherit;
  font-size: inherit;
  line-height: inherit;
}
button::after {
  border: none;
}

/* ---------- 主题变量：由 AppShell 根节点上的 class 切换 ---------- */
.app {
  --bg: #f7f3ec;
  --bg-soft: #efe7db;
  --surface: #ffffff;
  --text: #2a241d;
  --text-muted: #776c5e;
  --border: #e4dccd;
  --accent: #d96c4b;
  --accent-hover: #c25a3b;
  --accent-2: #2f5d62;
  --accent-3: #e9c46a;
  --accent-soft: #f7e0d7;
  --success: #2f9e6e;
  --danger: #c0392b;
  --shadow-sm: 0 1px 3px rgba(46, 38, 26, 0.08);
  --shadow: 0 6px 24px rgba(46, 38, 26, 0.08);
  --shadow-lg: 0 16px 48px rgba(46, 38, 26, 0.14);
  --radius: 16px;
  --radius-sm: 10px;
  --maxw: 1120px;
  --font-head: 'Noto Serif SC', 'Songti SC', 'SimSun', serif;

  min-height: 100vh;
  display: flex;
  flex-direction: column;
  background: var(--bg);
  color: var(--text);
  transition: background 0.3s ease, color 0.3s ease;
}

.app.theme-dark {
  --bg: #171410;
  --bg-soft: #201b15;
  --surface: #221d17;
  --text: #f0e9dd;
  --text-muted: #a99d8b;
  --border: #382f24;
  --accent: #e8855f;
  --accent-hover: #f29a76;
  --accent-2: #74a7a6;
  --accent-3: #e9c46a;
  --accent-soft: #3a251c;
  --shadow-sm: 0 1px 3px rgba(0, 0, 0, 0.4);
  --shadow: 0 8px 28px rgba(0, 0, 0, 0.45);
  --shadow-lg: 0 18px 52px rgba(0, 0, 0, 0.55);
}

.page {
  flex: 1;
}

.container {
  width: 92%;
  max-width: 1120px;
  margin: 0 auto;
}

.section {
  padding: 56px 0;
}

/* ---------- 区块标题 ---------- */
.section-head {
  text-align: center;
  margin-bottom: 40px;
}
.section-head .eyebrow {
  color: var(--accent);
  letter-spacing: 0.18em;
  text-transform: uppercase;
  font-size: 12px;
  font-weight: 600;
}
.section-head .section-head__title {
  display: block;
  font-family: var(--font-head);
  font-size: 28px;
  font-weight: 700;
  line-height: 1.3;
  margin-top: 6px;
}
.section-head .section-head__sub {
  display: block;
  color: var(--text-muted);
  margin-top: 10px;
  font-size: 15px;
}

/* ---------- 按钮 ---------- */
.btn {
  display: inline-flex;
  align-items: center;
  justify-content: center;
  padding: 14px 28px;
  border: 1px solid transparent;
  border-radius: 999px;
  font-weight: 600;
  font-size: 15px;
  line-height: 1.2;
  transition: background 0.2s ease, color 0.2s ease, box-shadow 0.2s ease;
}
.btn--primary {
  background: var(--accent);
  color: #fff;
}
.btn--ghost {
  background: transparent;
  color: var(--text);
  border-color: var(--border);
}
.btn--soft {
  background: var(--accent-soft);
  color: var(--accent);
}
.btn--light {
  color: #fff;
  border-color: rgba(255, 255, 255, 0.5);
}
.btn--block {
  display: flex;
  width: 100%;
}

/* ---------- 卡片 ---------- */
.card {
  background: var(--surface);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  box-shadow: var(--shadow-sm);
  overflow: hidden;
  transition: box-shadow 0.2s ease;
}

/* ---------- 徽章 / 标签 ---------- */
.chip {
  display: inline-flex;
  align-items: center;
  justify-content: center;
  padding: 6px 16px;
  border-radius: 999px;
  border: 1px solid var(--border);
  background: var(--surface);
  color: var(--text-muted);
  font-size: 13px;
  line-height: 1.4;
  transition: all 0.15s ease;
}
.chip--active {
  background: var(--accent);
  border-color: var(--accent);
  color: #fff;
}

.tag {
  display: inline-block;
  padding: 2px 10px;
  border-radius: 999px;
  background: var(--accent-soft);
  color: var(--accent);
  font-size: 12px;
  font-weight: 600;
  line-height: 1.6;
}

/* ---------- 表单 ---------- */
.field {
  display: block;
  margin-bottom: 18px;
}
.field__label {
  display: block;
  font-size: 14px;
  font-weight: 600;
  margin-bottom: 6px;
}
.field__input {
  width: 100%;
  height: 46px;
  padding: 0 14px;
  border: 1px solid var(--border);
  border-radius: var(--radius-sm);
  background: var(--surface);
  color: var(--text);
  font-size: 15px;
}
.field__textarea {
  width: 100%;
  min-height: 120px;
  padding: 12px 14px;
  border: 1px solid var(--border);
  border-radius: var(--radius-sm);
  background: var(--surface);
  color: var(--text);
  font-size: 15px;
}
.field__error {
  display: block;
  color: var(--danger);
  font-size: 12px;
  margin-top: 4px;
}

/* ---------- 页面标题 ---------- */
.page-head {
  text-align: center;
  margin-bottom: 32px;
}
.page-head .eyebrow {
  display: block;
  color: var(--accent);
  letter-spacing: 0.18em;
  font-weight: 600;
  font-size: 12px;
}
.page-head .page-head__title {
  display: block;
  font-family: var(--font-head);
  font-size: 32px;
  font-weight: 700;
  line-height: 1.3;
  margin-top: 6px;
}
.page-head .page-head__sub {
  display: block;
  color: var(--text-muted);
  margin-top: 8px;
  font-size: 14px;
}
</style>
```

## 第 3 章　组合式函数 composables

这四个文件是项目的「共享逻辑层」。它们的共同点是：**把状态定义在函数外面（模块级）**，
这样所有页面 import 到的是同一个 `ref`，实现跨页面共享。

> 如果写在 `useXxx()` 函数内部，每次调用都会创建一个新 `ref`，页面之间就不同步了。

### 3.1 useSite.js

**作用**：提供站点信息（名称、slogan、邮箱）。

- `site` 用 `reactive` 定义，是一个响应式对象。
- `provideSite()` 必须在**页面**的 `setup` 中调用，把 `site` 提供给后代组件。
- 子组件用 `inject('site')` 获取，例如 `NavBar`、`AppFooter`。
- 这是 Vue 的 **provide / inject**，用于跨层级传值，避免 props 一层层往下传。


```js
// 知识点：provide / inject —— 在页面组件中提供站点配置，供任意后代组件注入
import { reactive, provide } from 'vue'

export const site = reactive({
  name: '山海行记',
  slogan: '把世界，装进手账。',
  email: 'hello@wanderlog.dev',
})

export function provideSite() {
  provide('site', site)
}
```

### 3.2 useTheme.js

**作用**：明暗主题的共享状态 + 持久化 + 原生界面同步。

- `theme` 是模块级 `ref`，初始值从 `uni.getStorageSync('wanderlog-theme')` 读取。
- `watch(theme, ..., { immediate: true })`：主题一变就写入本地存储。
- `applyThemeToNative()`：通过条件编译 `// #ifndef H5`，在非 H5 端同步
  导航栏、页面背景、tabBar 的颜色。H5 端这段代码会被编译掉。
- `useTheme()` 返回 `{ theme, toggleTheme }`。

⚠️ 条件编译的注释**必须**是 `// #ifndef H5` / `// #endif` 这种固定格式，
它不是普通注释，而是编译器指令。


```js
// 知识点：组合式函数、ref、watch、uni 本地存储持久化、原生导航栏同步
import { ref, watch } from 'vue'

const STORAGE_KEY = 'wanderlog-theme'

function loadTheme() {
  try {
    return uni.getStorageSync(STORAGE_KEY) || 'light'
  } catch (e) {
    return 'light'
  }
}

const theme = ref(loadTheme())

export function applyThemeToNative() {
  // #ifndef H5
  const dark = theme.value === 'dark'
  try {
    uni.setNavigationBarColor({
      frontColor: dark ? '#ffffff' : '#000000',
      backgroundColor: dark ? '#171410' : '#F7F3EC',
    })
    uni.setBackgroundColor({
      backgroundColor: dark ? '#171410' : '#F7F3EC',
      backgroundColorTop: dark ? '#171410' : '#F7F3EC',
      backgroundColorBottom: dark ? '#171410' : '#F7F3EC',
    })
    uni.setTabBarStyle({
      color: dark ? '#A99D8B' : '#776C5E',
      selectedColor: dark ? '#E8855F' : '#D96C4B',
      backgroundColor: dark ? '#221D17' : '#FFFFFF',
      borderStyle: dark ? 'white' : 'black',
    })
  } catch (e) {
    // 某些平台不支持，忽略即可
  }
  // #endif
}

watch(
  theme,
  (value) => {
    try {
      uni.setStorageSync(STORAGE_KEY, value)
    } catch (e) {
      // 忽略存储异常
    }
    applyThemeToNative()
  },
  { immediate: true }
)

export function useTheme() {
  const toggleTheme = () => {
    theme.value = theme.value === 'light' ? 'dark' : 'light'
  }
  return { theme, toggleTheme }
}
```

### 3.3 useFavorites.js

**作用**：收藏列表的共享状态与持久化。

- `favorites` 是模块级 `ref`，存目的地 id 的数组。
- `load()` 从 `uni.getStorageSync` 读取；存储里可能是字符串，也可能是数组，做了兼容。
- `watch(..., { deep: true })`：数组增删都会触发，自动写回本地存储。
- `isFavorite(id)` 判断是否已收藏；`toggleFavorite(id)` 收藏/取消。
- 因为状态是模块级单例，首页卡片、列表页、详情页、NavBar 徽标会同步更新。

⚠️ 这里用 `JSON.stringify` 存、`JSON.parse` 取；如果存储的是对象/数组，建议保持这个习惯。


```js
// 知识点：模块级单例响应式状态（跨页面共享）、watch 深度监听、uni 本地存储持久化
import { ref, watch } from 'vue'

const STORAGE_KEY = 'wanderlog-favorites'

function load() {
  try {
    const raw = uni.getStorageSync(STORAGE_KEY)
    if (!raw) return []
    return typeof raw === 'string' ? JSON.parse(raw) : raw
  } catch (e) {
    return []
  }
}

const favorites = ref(load())

watch(
  favorites,
  (value) => {
    try {
      uni.setStorageSync(STORAGE_KEY, JSON.stringify(value))
    } catch (e) {
      // 忽略存储异常
    }
  },
  { deep: true }
)

export function useFavorites() {
  const isFavorite = (id) => favorites.value.includes(id)

  const toggleFavorite = (id) => {
    const index = favorites.value.indexOf(id)
    if (index >= 0) favorites.value.splice(index, 1)
    else favorites.value.push(id)
  }

  return { favorites, isFavorite, toggleFavorite }
}
```

### 3.4 usePageScroll.js

**作用**：根据页面滚动距离，控制「返回顶部」按钮是否显示。

- `onPageScroll` 是 uni-app 的**页面生命周期**，只能在页面 `setup` 中注册，
  不能写在普通组件里，所以封装成组合式函数。
- 页面调用 `usePageScroll()` 得到 `showBackTop`，再传给 `AppShell` → `BackToTop`。
- 阈值默认 400px：`e.scrollTop > 400` 时显示按钮。

⚠️ 普通 Vue 组件里没有 `onPageScroll`；如果一定要在组件里监听滚动，
需要页面把滚动值通过 props 传下来（本项目就是这么做的）。


```js
// 知识点：uni-app 页面生命周期 —— onPageScroll 只能在页面 setup 中注册，
// 因此封装为组合式函数，由各页面调用，再把结果传给 AppShell / BackToTop。
import { ref } from 'vue'
import { onPageScroll } from '@dcloudio/uni-app'

export function usePageScroll(threshold = 400) {
  const showBackTop = ref(false)

  onPageScroll((e) => {
    showBackTop.value = (e.scrollTop || 0) > threshold
  })

  return { showBackTop }
}
```

## 第 4 章　数据层 destinations.js

**作用**：模拟后端接口，提供 10 个目的地。

- 顶部 `img(seed, w, h)` 是一个小工厂函数，用 `picsum.photos` 生成占位图 URL。
- 每个目的地对象的字段如下：

| 字段 | 含义 | 用在哪 |
| --- | --- | --- |
| `id` | 唯一标识 | 详情页 query、收藏、列表 key |
| `name` / `country` / `region` | 名称、国家、大洲 | 卡片、搜索、筛选 |
| `rating` | 评分 0–5 | 星级、排序 |
| `tags` | 标签数组 | 标签胶囊、搜索 |
| `featured` | 是否精选 | 首页只取前 3 个 |
| `summary` / `description` | 简介 / 详情 | 卡片 / 详情页 |
| `image` / `gallery` | 封面 / 图集 | 卡片 / 详情图集 |
| `facts` | 实用信息对象 | 详情页 FactList |

下面是文件开头和一个完整数据对象（其余 9 个结构相同）：

```js
const img = (seed, w = 800, h = 600) => `https://picsum.photos/seed/${seed}/${w}/${h}`

export const destinations = [
  {
    id: 'fuji',
    name: '富士山',
    country: '日本',
    region: '亚洲',
    rating: 4.9,
    tags: ['自然', '徒步', '摄影'],
    featured: true,
    summary: '日本最高峰，四季皆景的圣岳。',
    description: '富士山海拔 3776 米，是日本的象征……',
    image: img('fuji'),
    gallery: [img('fuji', 1200, 800), img('fuji-2'), img('fuji-3'), img('fuji-4')],
    facts: { 最佳季节: '7–9 月登山，11–2 月赏雪', 建议天数: '3–5 天', 预算: '中等', 语言: '日语' },
  },
  // …… 其余 9 个目的地结构相同
]
```

💡 想加一个目的地，只需要往数组里追加一个同结构的对象；
首页、列表、统计、搜索会自动多出这一条。


## 第 5 章　五个页面

页面和组件的区别：**页面**在 `pages.json` 注册，可以被路由跳转；**组件**只能被页面或
其他组件引用。每个页面都用 `AppShell` 包起来，保证导航栏、页脚、返回顶部一致。

### 5.1 首页 pages/index/index.vue

**职责**：品牌头图、统计数字、功能卖点、精选目的地、行动按钮。

关键逻辑：

- `provideSite()`：向 NavBar / Footer 提供站点信息。
- `usePageScroll()`：得到 `showBackTop`，传给 `AppShell`。
- `useFavorites()`：得到收藏判断和切换函数。
- `featured` 是 `computed`：筛选 `featured === true` 的前 3 条。
- `features` 是写死的 3 张功能卡数据。
- `goDestinations()` / `goPlanner()`：用 `uni.switchTab` 跳 tab 页。
- 模板顺序：`HeroBanner` → `StatsBar` → 功能卡 → 精选目的地 → CTA。

⚠️ 首页里的按钮跳转必须用 `switchTab`，因为目的地和行程规划都是 tab 页；
用 `navigateTo` 跳 tab 页会失败。


```vue
<script setup>
// 知识点：computed、v-for、组件组合、组合式函数复用、provide
import { computed } from 'vue'
import AppShell from '../../components/AppShell.vue'
import HeroBanner from '../../components/HeroBanner.vue'
import StatsBar from '../../components/StatsBar.vue'
import SectionTitle from '../../components/SectionTitle.vue'
import FeatureCard from '../../components/FeatureCard.vue'
import DestinationCard from '../../components/DestinationCard.vue'
import BaseButton from '../../components/BaseButton.vue'
import { destinations } from '../../data/destinations'
import { useFavorites } from '../../composables/useFavorites'
import { usePageScroll } from '../../composables/usePageScroll'
import { provideSite } from '../../composables/useSite'

provideSite()
const { showBackTop } = usePageScroll()
const { isFavorite, toggleFavorite } = useFavorites()

const featured = computed(() => destinations.filter((d) => d.featured).slice(0, 3))

const features = [
  { icon: '🗺️', title: '灵感目的地', desc: '精选全球热门目的地，按大洲筛选，快速找到心之所向。' },
  { icon: '📝', title: '行程规划', desc: '用待办清单规划每一天，进度条实时反馈，本地自动保存。' },
  { icon: '♥️', title: '收藏夹', desc: '一键收藏心动风景，导航栏徽标实时显示收藏数量。' },
]

function goDestinations() {
  uni.switchTab({ url: '/pages/destinations/index' })
}

function goPlanner() {
  uni.switchTab({ url: '/pages/planner/index' })
}
</script>

<template>
  <AppShell :show-back-top="showBackTop">
    <HeroBanner>
      <template #footer>
        <text class="hero-note">已收录 {{ destinations.length }} 个目的地 · 持续更新中</text>
      </template>
    </HeroBanner>
    <StatsBar />

    <view class="section">
      <view class="container">
        <SectionTitle
          eyebrow="WHY WANDERLOG"
          title="为什么选择山海行记"
          subtitle="从灵感到出发，一站式记录你的旅行计划。"
        />
        <view class="feature-grid">
          <view v-for="f in features" :key="f.title" class="grid-col">
            <FeatureCard v-bind="f" />
          </view>
        </view>
      </view>
    </view>

    <view class="section section--alt">
      <view class="container">
        <SectionTitle
          eyebrow="FEATURED"
          title="精选目的地"
          subtitle="编辑团队为你甄选的三处此生必去之地。"
        />
        <view class="featured-grid">
          <view v-for="d in featured" :key="d.id" class="grid-col">
            <DestinationCard
              :destination="d"
              :favorite="isFavorite(d.id)"
              @toggle-favorite="toggleFavorite(d.id)"
            />
          </view>
        </view>
        <view class="center">
          <view class="btn btn--soft" @click="goDestinations">
            <text>查看全部目的地 →</text>
          </view>
        </view>
      </view>
    </view>

    <view class="section">
      <view class="container">
        <view class="cta">
          <text class="cta__title">准备好出发了吗？</text>
          <text class="cta__desc">现在就开始规划你的下一段旅程。</text>
          <BaseButton variant="primary" @click="goPlanner">
            <template #icon><text>✈️</text></template>
            <text>开始规划行程</text>
          </BaseButton>
        </view>
      </view>
    </view>
  </AppShell>
</template>
```

### 5.2 目的地列表 pages/destinations/index.vue

**职责**：搜索 + 按大洲筛选 + 排序。

关键逻辑：

1. `keyword`、`region`、`sort` 三个 `ref` 保存筛选条件。
2. `regions`：用 `Set` 对 `region` 去重，得到筛选胶囊。
3. `filtered` 是核心 `computed`：
   - 关键词匹配 `name` / `country` / `region` / `tags`；
   - 大洲筛选；
   - 按评分或名称排序（名称用 `localeCompare(..., 'zh')` 支持中文排序）。
4. `SearchBar` 用 `v-model` 绑定 `keyword`。
5. `FilterBar` 用两个命名 `v-model`：`v-model:region`、`v-model:sort`。
6. 有结果渲染 `DestinationList`，无结果渲染 `EmptyState`。

🧠 这是「**数据驱动视图**」最典型的例子：不操作 DOM，只改 `keyword`，
`filtered` 自动重算，列表自动重渲染。


```vue
<script setup>
// 知识点：ref、computed（联动筛选/搜索/排序）、组件组合、v-if/v-else
import { ref, computed } from 'vue'
import AppShell from '../../components/AppShell.vue'
import SearchBar from '../../components/SearchBar.vue'
import FilterBar from '../../components/FilterBar.vue'
import DestinationList from '../../components/DestinationList.vue'
import EmptyState from '../../components/EmptyState.vue'
import { destinations } from '../../data/destinations'
import { useFavorites } from '../../composables/useFavorites'
import { usePageScroll } from '../../composables/usePageScroll'
import { provideSite } from '../../composables/useSite'

provideSite()
const { showBackTop } = usePageScroll()

const keyword = ref('')
const region = ref('全部')
const sort = ref('default')
const { isFavorite, toggleFavorite } = useFavorites()

const regions = computed(() => [...new Set(destinations.map((d) => d.region))])

const filtered = computed(() => {
  const kw = keyword.value.trim()
  let list = destinations.filter((d) => {
    const matchKw =
      !kw ||
      d.name.includes(kw) ||
      d.country.includes(kw) ||
      d.region.includes(kw) ||
      d.tags.some((t) => t.includes(kw))
    const matchRegion = region.value === '全部' || d.region === region.value
    return matchKw && matchRegion
  })
  if (sort.value === 'rating') list = [...list].sort((a, b) => b.rating - a.rating)
  if (sort.value === 'name') list = [...list].sort((a, b) => a.name.localeCompare(b.name, 'zh'))
  return list
})

function clearFilters() {
  keyword.value = ''
  region.value = '全部'
}
</script>

<template>
  <AppShell :show-back-top="showBackTop">
    <view class="section">
      <view class="container">
        <view class="page-head">
          <text class="eyebrow">DESTINATIONS</text>
          <text class="page-head__title">探索目的地</text>
          <text class="page-head__sub">{{ filtered.length }} 个目的地 · 支持搜索、按大洲筛选与排序</text>
        </view>

        <view class="toolbar">
          <SearchBar v-model="keyword" />
          <FilterBar v-model:region="region" v-model:sort="sort" :regions="regions" />
        </view>

        <DestinationList
          v-if="filtered.length"
          :destinations="filtered"
          :is-favorite="isFavorite"
          @toggle-favorite="toggleFavorite"
        />
        <EmptyState v-else icon="🔍" title="没有找到匹配的目的地" desc="换个关键词或筛选条件试试吧。">
          <view class="btn btn--soft" @click="clearFilters">
            <text>清空筛选</text>
          </view>
        </EmptyState>
      </view>
    </view>
  </AppShell>
</template>
```

### 5.3 目的地详情 pages/destination/detail.vue

**职责**：展示单个目的地的图集、评分、标签、介绍、实用信息。

关键逻辑：

- `onLoad(options)`：接收列表页 `navigateTo` 传来的 `?id=fuji`。
  这是小程序 / App 获取页面参数的标准方式，**不是** `route.params`。
- `dest` 是 `computed`：根据 id 从数据里 `find`。
- `watch(dest)`：找到后设置原生导航栏标题，并滚回顶部。
- `goBack()`：先检查 `getCurrentPages().length`，有上一页就 `navigateBack`，
  否则 `switchTab` 回列表页。这样直接打开详情页时也不会卡死。
- 找不到 id 时渲染 `EmptyState`「目的地不存在」。

⚠️ `onLoad` 只在页面创建时执行一次；如果从详情页再跳另一个详情页，
需要用 `onLoad` 重新取参，或者用 `onShow` + 监听参数变化。


```vue
<script setup>
// 知识点：页面参数 onLoad、computed、watch、v-if/v-else、组件复用
import { ref, computed, watch } from 'vue'
import { onLoad } from '@dcloudio/uni-app'
import AppShell from '../../components/AppShell.vue'
import PhotoGallery from '../../components/PhotoGallery.vue'
import RatingStars from '../../components/RatingStars.vue'
import TagBadge from '../../components/TagBadge.vue'
import FactList from '../../components/FactList.vue'
import FavoriteButton from '../../components/FavoriteButton.vue'
import EmptyState from '../../components/EmptyState.vue'
import { destinations } from '../../data/destinations'
import { useFavorites } from '../../composables/useFavorites'
import { usePageScroll } from '../../composables/usePageScroll'
import { provideSite } from '../../composables/useSite'

provideSite()
const { showBackTop } = usePageScroll()
const { isFavorite, toggleFavorite } = useFavorites()

const id = ref('')

// 小程序 / App 通过 onLoad(options) 接收 navigateTo 的 query 参数
onLoad((options) => {
  id.value = (options && options.id) || ''
})

const dest = computed(() => destinations.find((d) => d.id === id.value))

watch(dest, (value) => {
  if (value) {
    uni.setNavigationBarTitle({ title: value.name })
    uni.pageScrollTo({ scrollTop: 0, duration: 0 })
  }
})

function goBack() {
  const pages = getCurrentPages()
  if (pages.length > 1) {
    uni.navigateBack()
  } else {
    uni.switchTab({ url: '/pages/destinations/index' })
  }
}

function goList() {
  uni.switchTab({ url: '/pages/destinations/index' })
}
</script>

<template>
  <AppShell :show-back-top="showBackTop">
    <view class="section">
      <view class="container">
        <template v-if="dest">
          <view class="crumb" @click="goBack">
            <text class="crumb__link">← 返回目的地列表</text>
          </view>

          <view class="detail">
            <view class="detail__gallery">
              <PhotoGallery :images="dest.gallery" />
            </view>

            <view class="detail__body card">
              <view class="detail__head">
                <view class="detail__title-wrap">
                  <text class="detail__name">{{ dest.name }}</text>
                  <text class="detail__country">{{ dest.country }} · {{ dest.region }}</text>
                </view>
                <FavoriteButton
                  :active="isFavorite(dest.id)"
                  @toggle="toggleFavorite(dest.id)"
                />
              </view>

              <RatingStars :model-value="dest.rating" readonly />

              <view class="detail__tags">
                <TagBadge v-for="t in dest.tags" :key="t" :text="t" />
              </view>

              <text class="detail__desc">{{ dest.description }}</text>

              <text class="detail__sub">实用信息</text>
              <FactList :facts="dest.facts" />
            </view>
          </view>
        </template>

        <EmptyState v-else icon="🧭" title="目的地不存在" desc="你访问的地址可能已失效。">
          <view class="btn btn--primary" @click="goList">
            <text>返回目的地列表</text>
          </view>
        </EmptyState>
      </view>
    </view>
  </AppShell>
</template>
```

### 5.4 行程规划 pages/planner/index.vue

**职责**：待办清单的增删改查 + 完成进度 + 本地持久化。

关键逻辑：

- `items`：清单数组；`filter`：`all` / `todo` / `done`。
- `onMounted`：从 `uni.getStorageSync('wanderlog-planner')` 读取。
- `watch(items, ..., { deep: true })`：任何增删改都自动写回存储。
- `add` / `remove` / `toggle` / `update`：四个操作函数。
- `inspire()`：从 6 条预设建议里随机挑一条加入清单。
- `filtered`、`doneCount`、`progress`：三个 `computed` 派生数据。
- 子组件：`PlannerForm` 负责输入，`PlannerItem` 负责单条显示，`ProgressBar` 负责进度。

🧠 这里的持久化是「**状态一变就保存**」，不需要手动点保存按钮。


```vue
<script setup>
// 知识点：ref、computed、watch 深度持久化、onMounted、v-for、v-if/v-else
import { ref, computed, watch, onMounted } from 'vue'
import AppShell from '../../components/AppShell.vue'
import PlannerForm from '../../components/PlannerForm.vue'
import PlannerItem from '../../components/PlannerItem.vue'
import ProgressBar from '../../components/ProgressBar.vue'
import EmptyState from '../../components/EmptyState.vue'
import { usePageScroll } from '../../composables/usePageScroll'
import { provideSite } from '../../composables/useSite'

provideSite()
const { showBackTop } = usePageScroll()

const STORAGE_KEY = 'wanderlog-planner'
const items = ref([])
const filter = ref('all')

const suggestions = [
  '兑换当地货币并通知银行',
  '预订机场往返接送',
  '购买旅行保险',
  '下载离线地图',
  '确认酒店入住时间',
  '准备常用药品',
]

function loadItems() {
  try {
    const raw = uni.getStorageSync(STORAGE_KEY)
    if (!raw) return []
    return typeof raw === 'string' ? JSON.parse(raw) : raw
  } catch (e) {
    return []
  }
}

onMounted(() => {
  items.value = loadItems()
})

// 深度监听：任何增删改都会自动写回 uni 本地存储
watch(
  items,
  (value) => {
    try {
      uni.setStorageSync(STORAGE_KEY, JSON.stringify(value))
    } catch (e) {
      // 忽略存储异常
    }
  },
  { deep: true }
)

function add(payload) {
  items.value.push({ id: Date.now(), text: payload.text, priority: payload.priority, done: false })
}
function remove(id) {
  items.value = items.value.filter((i) => i.id !== id)
}
function toggle(id) {
  const item = items.value.find((i) => i.id === id)
  if (item) item.done = !item.done
}
function update(id, text) {
  const item = items.value.find((i) => i.id === id)
  if (item) item.text = text
}
function inspire() {
  const pick = suggestions[Math.floor(Math.random() * suggestions.length)]
  add({ text: pick, priority: '低' })
}

const filtered = computed(() => {
  if (filter.value === 'done') return items.value.filter((i) => i.done)
  if (filter.value === 'todo') return items.value.filter((i) => !i.done)
  return items.value
})
const doneCount = computed(() => items.value.filter((i) => i.done).length)
const progress = computed(() =>
  items.value.length ? Math.round((doneCount.value / items.value.length) * 100) : 0
)

const filters = [
  { key: 'all', label: '全部' },
  { key: 'todo', label: '未完成' },
  { key: 'done', label: '已完成' },
]
</script>

<template>
  <AppShell :show-back-top="showBackTop">
    <view class="section">
      <view class="container planner">
        <view class="page-head">
          <text class="eyebrow">PLANNER</text>
          <text class="page-head__title">行程规划</text>
          <text class="page-head__sub">把出发前的琐事列成清单，逐项打勾，进度自动保存。</text>
        </view>

        <view class="planner__top card">
          <PlannerForm @add="add" />
          <view class="planner__bar">
            <ProgressBar :value="progress" />
            <text class="planner__count">{{ doneCount }} / {{ items.length }} 已完成</text>
          </view>
        </view>

        <view class="planner__toolbar">
          <view class="planner__filters">
            <view
              v-for="f in filters"
              :key="f.key"
              class="chip"
              :class="{ 'chip--active': filter === f.key }"
              @click="filter = f.key"
            >
              <text>{{ f.label }}</text>
            </view>
          </view>
          <view class="btn btn--soft" @click="inspire">
            <text>✨ 随机灵感</text>
          </view>
        </view>

        <view v-if="filtered.length" class="planner__list">
          <PlannerItem
            v-for="item in filtered"
            :key="item.id"
            :item="item"
            @toggle="toggle(item.id)"
            @remove="remove(item.id)"
            @update="update(item.id, $event)"
          />
        </view>
        <EmptyState v-else icon="📝" title="清单空空如也" desc="添加第一条待办，或试试「随机灵感」。">
          <view class="btn btn--primary" @click="inspire">
            <text>给我一点灵感</text>
          </view>
        </EmptyState>
      </view>
    </view>
  </AppShell>
</template>
```

### 5.5 关于 pages/about/index.vue

**职责**：项目介绍、技术栈、知识点展示、留言表单。

关键逻辑：

- 直接 import `site` 和 `provideSite`，页面自己展示站点信息。
- `techs`、`features`：两组展示数据。
- `copyEmail()`：调用 `uni.setClipboardData` 复制邮箱，再用 `uni.showToast` 提示。
- `ContactForm`：留言表单组件。

⚠️ H5 端 `uni.setClipboardData` 依赖浏览器剪贴板权限；小程序端需要在用户点击事件里调用。


```vue
<script setup>
// 知识点：provide/inject（页面向下提供站点信息）、组件组合、剪贴板 API
import AppShell from '../../components/AppShell.vue'
import SectionTitle from '../../components/SectionTitle.vue'
import ContactForm from '../../components/ContactForm.vue'
import FeatureCard from '../../components/FeatureCard.vue'
import { site, provideSite } from '../../composables/useSite'
import { usePageScroll } from '../../composables/usePageScroll'

provideSite()
const { showBackTop } = usePageScroll()

const techs = [
  'uni-app（Vue 3 组合式 API / <script setup>）',
  'pages.json 路由 + 原生 tabBar',
  'HBuilderX 多端运行与发布',
  'CSS 变量 + 明暗主题',
]

const features = [
  { icon: '🔄', title: '响应式状态', desc: 'ref 与 reactive 驱动视图自动更新，computed 派生数据，watch 持久化。' },
  { icon: '🧩', title: '组件通信', desc: 'props / emit / defineModel / provide-inject / 插槽，覆盖组件间多种通信方式。' },
  { icon: '🛣️', title: '多端与路由', desc: '一套代码编译到 H5、微信小程序与 Android App，页面参数通过 onLoad 获取。' },
]

function copyEmail() {
  uni.setClipboardData({
    data: site.email,
    success: () => uni.showToast({ title: '邮箱已复制', icon: 'none' }),
  })
}
</script>

<template>
  <AppShell :show-back-top="showBackTop">
    <view class="section">
      <view class="container">
        <SectionTitle
          eyebrow="ABOUT"
          title="关于山海行记"
          subtitle="一个用 Vue 3 + uni-app 从零构建的旅行主题多端应用。"
        />

        <view class="about-grid">
          <view class="about__text card">
            <text class="about__p">
              「山海行记」最初是一个 Vue 3 + Vite 的纯 Web 项目，本次实验把它迁移为 uni-app 工程：
              同一套组件与组合式函数，可用于 H5、微信小程序和 Android App。
            </text>
            <text class="about__p">
              项目包含 5 个页面与 24 个业务组件，覆盖官方教程从「声明式渲染」到「组合式函数」的绝大多数知识点。
            </text>
            <text class="about__p">所有数据均为前端模拟，本地存储用于持久化主题、收藏与行程清单。</text>
            <view class="about__contact">
              <text>📮 联系邮箱：</text>
              <text class="about__mail" @click="copyEmail">{{ site.email }}</text>
            </view>
          </view>

          <view class="about__tech card">
            <text class="about__tech-title">技术栈</text>
            <view v-for="t in techs" :key="t" class="about__list-item">
              <text class="about__dot">·</text>
              <text class="about__list-text">{{ t }}</text>
            </view>
          </view>
        </view>
      </view>
    </view>

    <view class="section section--alt">
      <view class="container">
        <SectionTitle eyebrow="FEATURES" title="它演示了哪些知识点" />
        <view class="feature-grid">
          <view v-for="f in features" :key="f.title" class="grid-col">
            <FeatureCard v-bind="f" />
          </view>
        </view>
      </view>
    </view>

    <view class="section">
      <view class="container">
        <SectionTitle
          eyebrow="CONTACT"
          title="给我留言"
          subtitle="演示表单双向绑定与校验，提交后仅做前端展示。"
        />
        <view class="contact-wrap">
          <ContactForm />
        </view>
      </view>
    </view>
  </AppShell>
</template>
```

## 第 6 章　二十五个组件

组件可以按职责分成 5 组：

| 分组 | 数量 | 组件 |
| --- | --- | --- |
| 布局外壳 | 4 | AppShell、NavBar、AppFooter、BackToTop |
| 通用展示 | 7 | SectionTitle、FeatureCard、TagBadge、EmptyState、BaseButton、RatingStars、ProgressBar |
| 目的地相关 | 9 | SearchBar、FilterBar、DestinationList、DestinationCard、HeroBanner、StatsBar、PhotoGallery、FactList、FavoriteButton |
| 表单与清单 | 4 | FormInput、ContactForm、PlannerForm、PlannerItem |
| 主题 | 1 | ThemeToggle |

### 组件通信方式速查

| 方式 | 用在哪 | 例子 |
| --- | --- | --- |
| props 父传子 | 几乎所有展示组件 | `:destination="d"` |
| emit 子传父 | 收藏、表单提交、清单操作 | `@toggle-favorite`、`@add` |
| `v-model` / `defineModel` | 输入框、筛选器 | `v-model="keyword"`、`v-model:region` |
| 插槽 slot | 按钮图标、空状态操作、头图补充 | `<template #icon>`、`<slot name="footer">` |
| provide / inject | 站点信息跨层级传递 | `site` |
| 模块级共享状态 | 主题、收藏 | `useTheme`、`useFavorites` |


### 6.1 布局外壳

> 说明：下面只展示 script 与 template，`<style scoped>` 请打开源文件查看。

#### AppShell.vue

**作用**：每个页面最外层的统一外壳。

- `NavBar` + 内容插槽 + `AppFooter` + `BackToTop`。
- `showBackTop` 由页面传入，控制返回顶部按钮。
- `themeClass` 是 `computed`，根据 `theme` 给根节点加 `theme-dark` / `theme-light`。
- `onMounted` 调用 `applyThemeToNative()`，每次进入新页面都同步原生导航栏。

⚠️ 这是「小程序没有全局布局容器」的解决方案：把布局做成组件，在每个页面手动包一层。

```vue
<script setup>
// 页面统一外壳：导航栏 + 内容区 + 页脚 + 返回顶部
// 小程序没有 App.vue 包裹页面的能力，因此用组件复用全局布局
import { computed, onMounted } from 'vue'
import { useTheme, applyThemeToNative } from '../composables/useTheme'
import NavBar from './NavBar.vue'
import AppFooter from './AppFooter.vue'
import BackToTop from './BackToTop.vue'

defineProps({
  showBackTop: { type: Boolean, default: false },
})

const { theme } = useTheme()
const themeClass = computed(() => (theme.value === 'dark' ? 'theme-dark' : 'theme-light'))

// 每次进入新页面时，把主题同步到原生导航栏
onMounted(() => applyThemeToNative())
</script>

<template>
  <view class="app" :class="themeClass">
    <NavBar />
    <view class="page">
      <slot />
    </view>
    <AppFooter />
    <BackToTop :visible="showBackTop" />
  </view>
</template>
```

#### NavBar.vue

**作用**：顶部品牌区 + 收藏入口 + 主题切换。

- `inject('site')` 拿到站点名称。
- `useFavorites()` 拿到收藏数组，`favoriteCount` 是 `computed`，实时显示数量徽标。
- 点击品牌回首页，点击心形去目的地列表（目前收藏没有独立页面）。
- 两个跳转都用 `uni.switchTab`。

⚠️ 这里没有用 `<RouterLink>`，因为 uni-app 没有 Vue Router。

```vue
<script setup>
// 知识点：inject、computed、v-if、点击导航
// uni-app 使用 pages.json 的 tabBar 完成底部导航，这里保留品牌区、收藏入口与主题切换
import { computed, inject } from 'vue'
import ThemeToggle from './ThemeToggle.vue'
import { useFavorites } from '../composables/useFavorites'

const site = inject('site')
const { favorites } = useFavorites()

const favoriteCount = computed(() => favorites.value.length)

function goHome() {
  uni.switchTab({ url: '/pages/index/index' })
}

function goDestinations() {
  uni.switchTab({ url: '/pages/destinations/index' })
}
</script>

<template>
  <view class="navbar">
    <view class="navbar__inner">
      <view class="navbar__brand" @click="goHome">
        <text class="navbar__logo">⛰</text>
        <text class="navbar__name">{{ site?.name }}</text>
      </view>

      <view class="navbar__actions">
        <view class="navbar__fav" @click="goDestinations">
          <text class="navbar__fav-icon">♥</text>
          <text v-if="favoriteCount" class="navbar__badge">{{ favoriteCount }}</text>
        </view>
        <ThemeToggle />
      </view>
    </view>
  </view>
</template>
```

#### AppFooter.vue

**作用**：页脚品牌、slogan、4 个 tab 链接、版权年份。

- `inject('site')` 取站点信息。
- `year` 是 `computed`，自动取当前年份。
- `links` 数组 + `v-for` 渲染，点击调用 `uni.switchTab`。
- 页脚 `margin-top: auto` 配合 `.app` 的 flex 布局，让短页面也能把页脚推到底部。

```vue
<script setup>
// 知识点：inject、computed、v-for、switchTab 导航
import { inject, computed } from 'vue'

const site = inject('site')
const year = computed(() => new Date().getFullYear())

const links = [
  { url: '/pages/index/index', label: '首页' },
  { url: '/pages/destinations/index', label: '目的地' },
  { url: '/pages/planner/index', label: '行程规划' },
  { url: '/pages/about/index', label: '关于' },
]

function go(url) {
  uni.switchTab({ url })
}
</script>

<template>
  <view class="footer">
    <view class="container footer__inner">
      <view class="footer__brand">
        <text class="footer__logo">⛰</text>
        <view>
          <text class="footer__name">{{ site?.name }}</text>
          <text class="footer__slogan">{{ site?.slogan }}</text>
        </view>
      </view>
      <view class="footer__links">
        <text v-for="l in links" :key="l.url" class="footer__link" @click="go(l.url)">{{ l.label }}</text>
      </view>
    </view>
    <view class="container footer__bottom">
      © {{ year }} {{ site?.name }} · 用 Vue 3 + uni-app 构建 · 仅供学习交流
    </view>
  </view>
</template>
```

#### BackToTop.vue

**作用**：右下角固定定位的「返回顶部」按钮。

- `visible` 由父页面传入（页面用 `usePageScroll` 算出来）。
- 点击调用 `uni.pageScrollTo({ scrollTop: 0 })`。
- `position: fixed` 在小程序和 H5 都支持。

⚠️ 它不知道自己滚了多远，只负责「显示/隐藏」和「滚回去」，滚动状态在页面层计算。

```vue
<script setup>
// 知识点：props、uni.pageScrollTo
// 小程序无法监听 window.scroll，滚动状态由页面通过 usePageScroll 计算后传入
defineProps({
  visible: { type: Boolean, default: false },
})

function toTop() {
  uni.pageScrollTo({ scrollTop: 0, duration: 300 })
}
</script>

<template>
  <view v-if="visible" class="backtop" @click="toTop">
    <text class="backtop__icon">↑</text>
  </view>
</template>
```

### 6.2 通用展示

> 说明：下面只展示 script 与 template，`<style scoped>` 请打开源文件查看。

#### SectionTitle.vue

**作用**：区块标题，统一「小标签 + 主标题 + 副标题」的排版。

- 三个 props：`eyebrow`、`title`、`subtitle`。
- 每个都用 `v-if` 判断，没传就不渲染。

```vue
<script setup>
// 知识点：props
defineProps({
  eyebrow: String,
  title: String,
  subtitle: String,
})
</script>

<template>
  <view class="section-head">
    <text v-if="eyebrow" class="eyebrow">{{ eyebrow }}</text>
    <text v-if="title" class="section-head__title">{{ title }}</text>
    <text v-if="subtitle" class="section-head__sub">{{ subtitle }}</text>
  </view>
</template>
```

#### FeatureCard.vue

**作用**：功能卖点卡片。

- props：`icon`、`title`、`desc`。
- 默认插槽 `<slot />` 可以追加自定义内容。
- 首页和关于页都用它渲染 3 张功能卡。

```vue
<script setup>
// 知识点：props、插槽
defineProps({
  icon: String,
  title: String,
  desc: String,
})
</script>

<template>
  <view class="card feature">
    <view class="feature__icon">
      <text>{{ icon }}</text>
    </view>
    <text class="feature__title">{{ title }}</text>
    <text class="feature__desc">{{ desc }}</text>
    <slot />
  </view>
</template>
```

#### TagBadge.vue

**作用**：单个标签胶囊，最简单的展示组件。

- 只有一个 `text` prop。
- 接收字符串，渲染成 `<text class="tag">`。

```vue
<script setup>
// 知识点：props
defineProps({ text: String })
</script>

<template>
  <text class="tag">{{ text }}</text>
</template>
```

#### EmptyState.vue

**作用**：空状态占位。

- props：`icon`（默认 🗺️）、`title`、`desc`。
- 默认插槽放操作按钮，例如「清空筛选」「返回列表」。
- 列表无结果、详情 id 不存在、行程清单为空时都用它。

```vue
<script setup>
// 知识点：props、插槽（默认插槽放置操作按钮）
defineProps({
  icon: { type: String, default: '🗺️' },
  title: String,
  desc: String,
})
</script>

<template>
  <view class="empty">
    <text class="empty__icon">{{ icon }}</text>
    <text class="empty__title">{{ title }}</text>
    <text v-if="desc" class="empty__desc">{{ desc }}</text>
    <view class="empty__action">
      <slot />
    </view>
  </view>
</template>
```

#### BaseButton.vue

**作用**：统一按钮。

- props：`variant`（primary / ghost / soft / light）、`block`（是否占满整行）。
- 两个插槽：具名 `icon` 和默认插槽。
- 点击时 `$emit('click', $event)`，把事件交给父组件。
- 用 `<button>` 而不是 `<view>`，保留小程序的点击语义和无障碍特性。

```vue
<script setup>
// 知识点：props、插槽（默认插槽 + 具名插槽 icon）、class 动态绑定
defineProps({
  variant: { type: String, default: 'primary' },
  block: { type: Boolean, default: false },
})
defineEmits(['click'])
</script>

<template>
  <button class="btn" :class="[`btn--${variant}`, { 'btn--block': block }]" @click="$emit('click', $event)">
    <slot name="icon" />
    <slot />
  </button>
</template>
```

#### RatingStars.vue

**作用**：星级评分。

- props：`modelValue`、`max`（默认 5）、`readonly`。
- `stars` 是 `computed`，生成 1..max 的数组。
- 可编辑时点击星星，`emit('update:modelValue', s)`，父组件可用 `v-model`。
- 本项目所有地方都传了 `readonly`，只做展示。
- 右侧还会显示数值（`Number(modelValue).toFixed(1)`）。

```vue
<script setup>
// 知识点：props、computed、事件（update:modelValue 实现 v-model）、v-for、class 绑定
import { computed } from 'vue'

const props = defineProps({
  modelValue: { type: Number, default: 0 },
  max: { type: Number, default: 5 },
  readonly: { type: Boolean, default: false },
})
const emit = defineEmits(['update:modelValue'])

const stars = computed(() => Array.from({ length: props.max }, (_, i) => i + 1))

function choose(s) {
  if (!props.readonly) emit('update:modelValue', s)
}
</script>

<template>
  <view class="stars">
    <text
      v-for="s in stars"
      :key="s"
      class="star"
      :class="{ 'star--on': s <= Math.round(modelValue) }"
      @click="choose(s)"
    >★</text>
    <text class="stars__value">{{ Number(modelValue).toFixed(1) }}</text>
  </view>
</template>
```

#### ProgressBar.vue

**作用**：进度条。

- 只有一个 `value` prop。
- `percent` 是 `computed`，用 `Math.min/max` 把值限制在 0–100。
- 填充宽度用内联 `:style="{ width: percent + '%' }"`，因为宽度是动态的。
- 用 `transition: width` 做平滑动画。

```vue
<script setup>
// 知识点：props、computed、内联样式绑定（:style）
import { computed } from 'vue'

const props = defineProps({
  value: { type: Number, default: 0 },
})

const percent = computed(() => Math.min(100, Math.max(0, props.value)))
</script>

<template>
  <view class="progress">
    <view class="progress__track">
      <view class="progress__fill" :style="{ width: percent + '%' }"></view>
    </view>
    <text class="progress__text">{{ percent }}%</text>
  </view>
</template>
```

### 6.3 目的地相关

> 说明：下面只展示 script 与 template，`<style scoped>` 请打开源文件查看。

#### SearchBar.vue

**作用**：搜索输入框。

- `defineModel` 实现组件级 `v-model`，父页面写 `v-model="keyword"`。
- 有内容时显示清除按钮，点击把 model 设为空。
- `focused` 控制聚焦。小程序没有 DOM 的 `input.focus()`，只能用 `<input :focus="focused">`。
- `defineExpose({ focus })` 把聚焦方法暴露给父组件（本项目暂未调用，作为知识点保留）。

⚠️ `:focus` 是「一次性」的：设为 true 后用户失焦，需要先设回 false 再设 true 才能再次聚焦。

```vue
<script setup>
// 知识点：defineModel（组件级 v-model）、defineExpose、v-if
// 小程序不能直接调用 DOM 的 focus()，改用 input 的 :focus 属性
import { ref } from 'vue'

const model = defineModel({ type: String, default: '' })
const focused = ref(false)

function focus() {
  focused.value = false
  setTimeout(() => {
    focused.value = true
  }, 50)
}

defineExpose({ focus })
</script>

<template>
  <view class="searchbar">
    <text class="searchbar__icon">🔍</text>
    <input
      v-model="model"
      type="text"
      class="searchbar__input"
      confirm-type="search"
      placeholder="搜索目的地、国家或标签…"
      placeholder-class="searchbar__placeholder"
      :focus="focused"
      @blur="focused = false"
    />
    <view v-if="model" class="searchbar__clear" @click="model = ''">
      <text>✕</text>
    </view>
  </view>
</template>
```

#### FilterBar.vue

**作用**：大洲筛选胶囊 + 排序下拉。

- props：`regions`（大洲数组）。
- 两个命名 `defineModel`：`region` 和 `sort`，对应父组件的 `v-model:region` / `v-model:sort`。
- 小程序没有 `<select>`，改用 `<picker mode="selector">`。
- `sortIndex` 是 `computed`，把当前排序值换算成 picker 的下标。
- `onSortChange(e)` 从 `e.detail.value` 取选中下标，再写回 `sort`。

```vue
<script setup>
// 知识点：props、defineModel（命名 v-model）、v-for、class 绑定
// 小程序没有 <select>，改用 uni-app 的 <picker mode="selector">
import { computed } from 'vue'

defineProps({
  regions: { type: Array, default: () => [] },
})

const region = defineModel('region', { default: '全部' })
const sort = defineModel('sort', { default: 'default' })

const sortOptions = [
  { value: 'default', label: '默认排序' },
  { value: 'rating', label: '按评分' },
  { value: 'name', label: '按名称' },
]
const sortLabels = sortOptions.map((o) => o.label)

const sortIndex = computed(() => Math.max(0, sortOptions.findIndex((o) => o.value === sort.value)))

function onSortChange(e) {
  sort.value = sortOptions[Number(e.detail.value)].value
}
</script>

<template>
  <view class="filterbar">
    <view class="filterbar__chips">
      <view
        v-for="r in ['全部', ...regions]"
        :key="r"
        class="chip"
        :class="{ 'chip--active': region === r }"
        @click="region = r"
      >
        <text>{{ r }}</text>
      </view>
    </view>

    <picker mode="selector" :range="sortLabels" :value="sortIndex" @change="onSortChange">
      <view class="filterbar__select">
        <text>{{ sortLabels[sortIndex] }} ▾</text>
      </view>
    </picker>
  </view>
</template>
```

#### DestinationList.vue

**作用**：目的地两列网格。

- props：`destinations`、`isFavorite`（函数）。
- 收到 `toggle-favorite` 后继续向上抛，并附带 id。
- 用 Flexbox 的 `width: 50%` 实现两列，窄屏媒体查询改成 100%。
- 这就是「容器组件」：只负责布局和事件透传，不关心单张卡片长什么样。

```vue
<script setup>
// 知识点：props、v-for（带 key）、事件透传
import DestinationCard from './DestinationCard.vue'

defineProps({
  destinations: { type: Array, default: () => [] },
  isFavorite: { type: Function, default: () => false },
})
defineEmits(['toggle-favorite'])
</script>

<template>
  <view class="dest-list">
    <view v-for="d in destinations" :key="d.id" class="dest-list__item">
      <DestinationCard
        :destination="d"
        :favorite="isFavorite(d.id)"
        @toggle-favorite="$emit('toggle-favorite', d.id)"
      />
    </view>
  </view>
</template>
```

#### DestinationCard.vue

**作用**：单张目的地卡片。

- props：`destination`、`favorite`。
- 点击图片区域调用 `uni.navigateTo` 跳详情，url 里带 `?id=xxx`。
- `FavoriteButton` 上用了 `@click.stop`（在按钮组件内部），防止点收藏时也触发跳转。
- `<image mode="aspectFill">`：保持比例裁剪填满，必须显式给宽高。
- 卡片里组合了 `TagBadge`、`RatingStars`、`FavoriteButton` 三个子组件。

```vue
<script setup>
// 知识点：props、事件、子组件组合、v-for、页面跳转（navigateTo）
import TagBadge from './TagBadge.vue'
import RatingStars from './RatingStars.vue'
import FavoriteButton from './FavoriteButton.vue'

const props = defineProps({
  destination: { type: Object, required: true },
  favorite: { type: Boolean, default: false },
})
defineEmits(['toggle-favorite'])

function goDetail() {
  uni.navigateTo({ url: `/pages/destination/detail?id=${props.destination.id}` })
}
</script>

<template>
  <view class="card dest-card">
    <view class="dest-card__media" @click="goDetail">
      <image class="dest-card__img" :src="destination.image" mode="aspectFill" />
      <view class="dest-card__fav">
        <FavoriteButton :active="favorite" @toggle="$emit('toggle-favorite')" />
      </view>
    </view>
    <view class="dest-card__body">
      <view class="dest-card__top">
        <text class="dest-card__name">{{ destination.name }}</text>
        <RatingStars :model-value="destination.rating" readonly />
      </view>
      <text class="dest-card__country">{{ destination.country }} · {{ destination.region }}</text>
      <text class="dest-card__summary">{{ destination.summary }}</text>
      <view class="dest-card__tags">
        <TagBadge v-for="t in destination.tags" :key="t" :text="t" />
      </view>
    </view>
  </view>
</template>
```

#### HeroBanner.vue

**作用**：首页头图。

- 用 `<image>` 承载背景，而不是 CSS `background-image`。
  原因：小程序不支持 CSS 网络背景图。
- 半透明遮罩 `.hero__overlay` 用 `linear-gradient` 让文字可读。
- 两个按钮用 `switchTab` 去目的地/行程页。
- `footer` 具名插槽：首页用它显示「已收录 10 个目的地」。
- 标题里的 `{{ '\n' }}` 配合 `white-space: pre-line` 实现强制换行。

```vue
<script setup>
// 知识点：插槽（具名插槽 footer）、页面跳转、image 组件
// 小程序不支持 CSS 网络背景图，因此改用 <image> 承载背景
const heroImage = 'https://picsum.photos/seed/wanderlog-hero/1600/900'

function go(url) {
  uni.switchTab({ url })
}
</script>

<template>
  <view class="hero">
    <image class="hero__bg" :src="heroImage" mode="aspectFill" />
    <view class="hero__overlay"></view>
    <view class="container hero__content">
      <text class="hero__eyebrow">WANDERLOG · 环球旅行手账</text>
      <text class="hero__title">把世界，{{ '\n' }}装进手账。</text>
      <text class="hero__desc">
        收录心动目的地、规划专属行程、记录每一次出发。用 Vue 3 打造的旅行灵感集，陪你走过山海。
      </text>
      <view class="hero__actions">
        <view class="btn btn--primary" @click="go('/pages/destinations/index')">
          <text>探索目的地</text>
        </view>
        <view class="btn btn--ghost btn--light" @click="go('/pages/planner/index')">
          <text>规划行程</text>
        </view>
      </view>
      <!-- 具名插槽 footer：由父页面提供补充说明 -->
      <slot name="footer" />
    </view>
  </view>
</template>
```

#### StatsBar.vue

**作用**：首页统计条。

- 直接 import 数据源，`stats` 是 `computed`：
  - 目的地数量 = 数组长度；
  - 国家数 / 大洲数 = `Set` 去重后的 size；
  - 平均评分 = 总分 / 数量，保留 1 位小数。
- 四列布局，窄屏变两列。

🧠 这些数字没有写死，数据一变就自动更新。

```vue
<script setup>
// 知识点：computed（从数据源派生出统计指标）、v-for
import { computed } from 'vue'
import { destinations } from '../data/destinations'

const stats = computed(() => {
  const countries = new Set(destinations.map((d) => d.country))
  const regions = new Set(destinations.map((d) => d.region))
  const avgRating = (destinations.reduce((s, d) => s + d.rating, 0) / destinations.length).toFixed(1)
  return [
    { label: '收录目的地', value: destinations.length },
    { label: '覆盖国家', value: countries.size },
    { label: '涉及大洲', value: regions.size },
    { label: '平均评分', value: avgRating },
  ]
})
</script>

<template>
  <view class="stats">
    <view class="container stats__grid">
      <view v-for="s in stats" :key="s.label" class="stats__item">
        <text class="stats__value">{{ s.value }}</text>
        <text class="stats__label">{{ s.label }}</text>
      </view>
    </view>
  </view>
</template>
```

#### PhotoGallery.vue

**作用**：详情页图集。

- prop：`images` 数组。
- `active` 保存当前大图下标。
- `prev()` / `next()` 用取模实现循环切换：
  `(active - 1 + len) % len`、`(active + 1) % len`。
- 只有一张图时用 `v-show` 隐藏左右箭头。
- 底部缩略图 `v-for` 渲染，点击直接切换。

```vue
<script setup>
// 知识点：props、ref、v-for、v-show、class 绑定
import { ref } from 'vue'

const props = defineProps({
  images: { type: Array, default: () => [] },
})

const active = ref(0)
const prev = () => {
  active.value = (active.value - 1 + props.images.length) % props.images.length
}
const next = () => {
  active.value = (active.value + 1) % props.images.length
}
</script>

<template>
  <view class="gallery">
    <view class="gallery__main">
      <image class="gallery__img" :src="images[active]" mode="aspectFill" />
      <view v-show="images.length > 1" class="gallery__nav gallery__nav--prev" @click="prev">
        <text>‹</text>
      </view>
      <view v-show="images.length > 1" class="gallery__nav gallery__nav--next" @click="next">
        <text>›</text>
      </view>
    </view>
    <view class="gallery__thumbs">
      <view
        v-for="(img, i) in images"
        :key="i"
        class="gallery__thumb"
        :class="{ 'gallery__thumb--active': i === active }"
        @click="active = i"
      >
        <image class="gallery__thumb-img" :src="img" mode="aspectFill" />
      </view>
    </view>
  </view>
</template>
```

#### FactList.vue

**作用**：把 `facts` 对象渲染成信息行。

- prop：`facts`（对象）。
- `v-for="(value, key) in facts"` 同时拿到键和值。
- 左侧固定宽度显示键，右侧显示值，中间用虚线分隔。

```vue
<script setup>
// 知识点：props、v-for 遍历对象
defineProps({
  facts: { type: Object, default: () => ({}) },
})
</script>

<template>
  <view class="facts">
    <view v-for="(value, key) in facts" :key="key" class="facts__row">
      <text class="facts__key">{{ key }}</text>
      <text class="facts__value">{{ value }}</text>
    </view>
  </view>
</template>
```

#### FavoriteButton.vue

**作用**：心形收藏按钮。

- prop：`active` 控制实心/空心。
- 点击 `$emit('toggle')`，具体收藏逻辑由页面处理。
- `@click.stop` 阻止事件冒泡，避免点收藏时触发卡片跳转。
- `:aria-label` 提升无障碍体验。

```vue
<script setup>
// 知识点：props、事件（emit）、class 动态绑定
defineProps({
  active: { type: Boolean, default: false },
})
defineEmits(['toggle'])
</script>

<template>
  <view
    class="fav"
    :class="{ 'fav--active': active }"
    :aria-label="active ? '取消收藏' : '收藏'"
    @click.stop="$emit('toggle')"
  >
    <text class="fav__icon">{{ active ? '♥' : '♡' }}</text>
  </view>
</template>
```

### 6.4 表单与清单

> 说明：下面只展示 script 与 template，`<style scoped>` 请打开源文件查看。

#### FormInput.vue

**作用**：通用输入项。

- props：`label`、`type`、`textarea`、`error`。
- `defineModel` 实现 `v-model`。
- `inputType` 是 `computed`：小程序 input 的 type 不支持 `email`，统一回退成 `text`。
- `textarea` 用 `:maxlength="-1"` 取消长度限制。
- 有 `error` 时在下方显示红色提示。

```vue
<script setup>
// 知识点：props、defineModel、v-if/v-else、v-model
// 小程序 input 的 type 仅支持 text/number/digit/idcard/nickname，email 统一回退为 text
import { computed } from 'vue'

const props = defineProps({
  label: String,
  type: { type: String, default: 'text' },
  textarea: { type: Boolean, default: false },
  error: String,
})

const model = defineModel({ default: '' })
const inputType = computed(() => (props.type === 'email' ? 'text' : props.type))
</script>

<template>
  <view class="field">
    <text class="field__label">{{ label }}</text>
    <textarea
      v-if="textarea"
      v-model="model"
      class="field__textarea"
      :maxlength="-1"
      placeholder="请输入…"
      placeholder-class="field__placeholder"
    ></textarea>
    <input
      v-else
      v-model="model"
      class="field__input"
      :type="inputType"
      placeholder="请输入…"
      placeholder-class="field__placeholder"
    />
    <text v-if="error" class="field__error">{{ error }}</text>
  </view>
</template>
```

#### ContactForm.vue

**作用**：留言表单。

- `form` 和 `errors` 都是 `reactive`，分别存输入值和错误提示。
- `emailValid` 是 `computed`，用正则判断邮箱格式。
- `validate()` 逐项校验，返回是否全部通过。
- `submitted` 控制「表单 / 成功页」切换。
- 成功页会回显用户输入，`reset()` 可以再写一条。
- 表单本身不会发请求，纯前端演示。

```vue
<script setup>
// 知识点：reactive、ref、computed、表单校验、组件 v-model、v-if/v-else
import { reactive, ref, computed } from 'vue'
import FormInput from './FormInput.vue'

const form = reactive({ name: '', email: '', message: '' })
const errors = reactive({ name: '', email: '', message: '' })
const submitted = ref(false)

const emailValid = computed(() => /^[^\s@]+@[^\s@]+\.[^\s@]+$/.test(form.email))

function validate() {
  errors.name = form.name.trim() ? '' : '请输入你的称呼'
  errors.email = form.email.trim() ? (emailValid.value ? '' : '邮箱格式不正确') : '请输入邮箱'
  errors.message = form.message.trim() ? '' : '请填写留言内容'
  return !errors.name && !errors.email && !errors.message
}

function submit() {
  if (validate()) submitted.value = true
}

function reset() {
  form.name = ''
  form.email = ''
  form.message = ''
  errors.name = ''
  errors.email = ''
  errors.message = ''
  submitted.value = false
}
</script>

<template>
  <view class="contact">
    <view v-if="!submitted" class="contact__form">
      <FormInput v-model="form.name" label="称呼" :error="errors.name" />
      <FormInput v-model="form.email" label="邮箱" type="email" :error="errors.email" />
      <FormInput v-model="form.message" label="留言" textarea :error="errors.message" />
      <view class="btn btn--primary btn--block" @click="submit">
        <text>提交留言</text>
      </view>
    </view>

    <view v-else class="contact__success">
      <text class="contact__success-icon">✅</text>
      <text class="contact__success-title">谢谢你的留言，{{ form.name }}！</text>
      <text class="contact__success-desc">这是一份前端演示，留言不会被真正发送。以下是你的内容：</text>
      <view class="contact__quote">
        <text>{{ form.message }}</text>
      </view>
      <view class="btn btn--soft" @click="reset">
        <text>再写一条</text>
      </view>
    </view>
  </view>
</template>
```

#### PlannerForm.vue

**作用**：添加行程的输入区。

- `text`、`priority`、`focused` 三个 `ref`。
- 优先级用 `<picker mode="selector">`，`priorityIndex` 是 `computed`。
- `submit()` 校验非空后 `emit('add', { text, priority })`，然后清空输入。
- 提交后用 `focused = false → true` 重新聚焦，方便连续添加。
- 键盘「完成」也会触发 `@confirm="submit"`。

```vue
<script setup>
// 知识点：ref、v-model、事件（emit）、表单输入、picker 选择器
import { ref, computed } from 'vue'

const emit = defineEmits(['add'])

const text = ref('')
const priority = ref('中')
const focused = ref(false)
const priorities = ['高', '中', '低']

const priorityIndex = computed(() => Math.max(0, priorities.indexOf(priority.value)))

function onPriorityChange(e) {
  priority.value = priorities[Number(e.detail.value)]
}

function submit() {
  if (!text.value.trim()) return
  emit('add', { text: text.value.trim(), priority: priority.value })
  text.value = ''
  // 小程序通过 :focus 控制聚焦，重新触发一次以获得输入焦点
  focused.value = false
  setTimeout(() => {
    focused.value = true
  }, 50)
}
</script>

<template>
  <view class="planner-form">
    <input
      v-model="text"
      class="planner-form__input"
      placeholder="例如：预订富士山脚下民宿…"
      placeholder-class="planner-form__placeholder"
      confirm-type="done"
      :focus="focused"
      @blur="focused = false"
      @confirm="submit"
    />
    <picker mode="selector" :range="priorities" :value="priorityIndex" @change="onPriorityChange">
      <view class="planner-form__priority">
        <text>{{ priority }}优先级 ▾</text>
      </view>
    </picker>
    <view class="btn btn--primary" @click="submit">
      <text>添加</text>
    </view>
  </view>
</template>
```

#### PlannerItem.vue

**作用**：单条行程。

- props：`item`；emits：`toggle`、`remove`、`update`。
- `editing` / `draft` 控制编辑状态和草稿。
- 复选框用 `<checkbox-group>` + `<checkbox>`，`@change` 抛 `toggle`。
- 优先级用 `priorityClass` 映射成不同颜色。
- 编辑时显示 input，确认后抛 `update`；取消修改则保留原文。
- 删除按钮抛 `remove`。

```vue
<script setup>
// 知识点：props、事件、ref、v-if/v-else、class 绑定、confirm 事件
import { ref } from 'vue'

const props = defineProps({
  item: { type: Object, required: true },
})
const emit = defineEmits(['toggle', 'remove', 'update'])

const editing = ref(false)
const draft = ref('')

const priorityClass = { 高: 'prio--high', 中: 'prio--mid', 低: 'prio--low' }

function startEdit() {
  draft.value = props.item.text
  editing.value = true
}

function save() {
  emit('update', draft.value.trim() || props.item.text)
  editing.value = false
}
</script>

<template>
  <view class="planner-item" :class="{ 'planner-item--done': item.done }">
    <checkbox-group class="planner-item__check" :value="item.done ? ['done'] : []" @change="emit('toggle')">
      <checkbox value="done" :checked="item.done" color="#d96c4b" />
    </checkbox-group>

    <text class="planner-item__prio" :class="priorityClass[item.priority]">{{ item.priority }}</text>

    <template v-if="editing">
      <input
        v-model="draft"
        class="planner-item__edit"
        confirm-type="done"
        :focus="true"
        @confirm="save"
      />
      <view class="planner-item__btn" @click="save"><text>✓</text></view>
    </template>
    <template v-else>
      <text class="planner-item__text">{{ item.text }}</text>
      <view class="planner-item__btn" @click="startEdit"><text>✎</text></view>
    </template>

    <view class="planner-item__btn planner-item__btn--danger" @click="emit('remove')">
      <text>🗑</text>
    </view>
  </view>
</template>
```

### 6.5 主题

> 说明：下面只展示 script 与 template，`<style scoped>` 请打开源文件查看。

#### ThemeToggle.vue

**作用**：明暗主题切换按钮。

- 调用 `useTheme()` 拿到 `theme` 和 `toggleTheme`。
- 根据主题显示 🌙 或 ☀️。
- 点击调用 `toggleTheme()`，模块级 `theme` 变化后：
  - `AppShell` 的 class 变化 → CSS 变量切换；
  - `useTheme` 的 `watch` → 写入存储；
  - 非 H5 端 → 同步原生导航栏。
- `aria-label` 根据当前主题动态变化。

```vue
<script setup>
// 知识点：组合式函数复用、v-if/v-else、事件处理
import { useTheme } from '../composables/useTheme'

const { theme, toggleTheme } = useTheme()
</script>

<template>
  <view class="theme-toggle" :aria-label="theme === 'light' ? '切换到深色模式' : '切换到浅色模式'" @click="toggleTheme">
    <text v-if="theme === 'light'">🌙</text>
    <text v-else>☀️</text>
  </view>
</template>
```

## 第 7 章　样式体系与多端兼容

### 7.1 三层样式

| 层级 | 位置 | 作用 |
| --- | --- | --- |
| 全局基础 | `App.vue` 的 `<style>` | `page` 背景、字体、button 重置、公共类 |
| 主题变量 | `App.vue` 的 `.app` / `.app.theme-dark` | 颜色、阴影、圆角等设计 token |
| 组件样式 | 各组件的 `<style scoped>` | 只影响当前组件，避免类名冲突 |

### 7.2 主题切换的核心代码

```css
.app {
  --bg: #f7f3ec;
  --text: #2a241d;
  --accent: #d96c4b;
}
.app.theme-dark {
  --bg: #171410;
  --text: #f0e9dd;
  --accent: #e8855f;
}
```

组件里只写 `var(--bg)`，不写具体颜色。切换根节点 class，所有组件一起变色。

### 7.3 响应式断点

本项目用两个断点：

| 断点 | 变化 |
| --- | --- |
| `max-width: 860px` | 首页三栏 → 一栏；详情两栏 → 上下；关于页两栏 → 上下 |
| `max-width: 720px` | 目的地两列 → 一列；统计四列 → 两列；头图标题字号变小 |

### 7.4 为小程序做的兼容处理

| Web 写法 | 问题 | 本项目做法 |
| --- | --- | --- |
| `color-mix()` | WXSS 不支持 | 改用固定 `rgba()` |
| `aspect-ratio` | 兼容性差 | 改用固定高度 + `mode="aspectFill"` |
| `backdrop-filter` | 小程序不支持 | 移除，改用半透明背景 |
| `<Transition>` | 小程序不支持 | 移除页面过渡动画 |
| `<select>` | 小程序没有 | `<picker mode="selector">` |
| `input.focus()` | 没有 DOM | `<input :focus="focused">` + 先 false 再 true |
| `background-image: url(网络图)` | 小程序不支持 | `<image>` 组件 + `mode="aspectFill"` |
| `html` / `document` | 逻辑层无 DOM | 数据驱动 + `uni.*` API |

### 7.5 关于 px 与 rpx

`manifest.json` 里设置了 `"transformPx": false`，所以样式里的 `px` 不会被转成 `rpx`，
在不同手机上尺寸保持一致，但不会随屏幕宽度等比放大。

如果想让布局在大屏手机上按比例缩放，可以把 `transformPx` 改为 `true`，
或者把关键尺寸改成 `rpx`（750rpx = 屏幕宽度）。本项目为了和 Web 版视觉一致，选择了固定 `px`。


## 第 8 章　三条完整数据流

看懂这三条链路，基本就理解了这个项目的架构。

### 8.1 收藏：组件事件 → 共享状态 → 持久化 → 多处更新

```text
用户点击 FavoriteButton
  └─ $emit('toggle')
       └─ DestinationCard / DestinationList 继续向上抛
            └─ 页面调用 toggleFavorite(id)
                 └─ useFavorites 的模块级 favorites 数组变化
                      ├─ watch → uni.setStorageSync 写入本地
                      ├─ NavBar 的 favoriteCount 重新计算
                      └─ 所有卡片的 active 状态重新渲染
```

涉及文件：`FavoriteButton.vue` → `DestinationCard.vue` → `DestinationList.vue`
→ `pages/destinations/index.vue` → `useFavorites.js` → `NavBar.vue`。

### 8.2 搜索：输入 → computed → 列表重渲染

```text
用户在 SearchBar 输入
  └─ defineModel 更新 keyword
       └─ 页面的 filtered computed 重新计算
            └─ DestinationList 收到新的 destinations 数组
                 └─ v-for 重新渲染卡片
```

整个过程没有手动操作 DOM，也没有手动调「刷新列表」函数。

### 8.3 行程：增删改 → 深度监听 → 自动保存

```text
PlannerForm 抛 add / PlannerItem 抛 toggle·update·remove
  └─ 页面修改 items 数组
       ├─ watch(items, { deep: true }) → uni.setStorageSync
       ├─ filtered / doneCount / progress 重新计算
       └─ ProgressBar、清单、空状态自动更新
```

🧠 这三条链路的共同点：**只改状态，视图和存储自动跟着变**。


## 第 9 章　Web → uni-app 迁移对照

| Web 版（vue-app） | uni-app 版 | 为什么 |
| --- | --- | --- |
| `div` / `p` / `span` / `h1` | `view` / `text` | 小程序没有 HTML 标签 |
| `img` | `image mode="aspectFill"` | 小程序图片组件必须显式宽高 |
| `RouterLink` / `router.push` | `uni.navigateTo` / `uni.switchTab` | uni-app 没有 Vue Router |
| Vue Router 路由表 | `pages.json` + `onLoad(options)` | 页面参数通过 query 传递 |
| `localStorage` | `uni.getStorageSync` / `uni.setStorageSync` | 跨端存储 API |
| `window.scrollY` | `onPageScroll` | 小程序没有 window 滚动事件 |
| `window.scrollTo` | `uni.pageScrollTo` | 跨端滚动 API |
| `<html data-theme>` | 根节点 `.theme-dark` + `uni.setNavigationBarColor` | 小程序没有 html 节点 |
| `<select>` | `<picker mode="selector">` | 小程序没有原生下拉框 |
| `inputRef.focus()` | `<input :focus="focused">` | 没有 DOM API |
| `color-mix()` / `aspect-ratio` / `backdrop-filter` | 固定 rgba / 固定高度 | WXSS 兼容性 |
| `<Transition>` | 移除 | 小程序不支持 Vue Transition |
| `main.js` 里 `createApp().mount()` | `createSSRApp` + 导出 `createApp` 工厂 | uni-app 运行时自己挂载 |

### 迁移时最容易踩的坑

1. **把 tab 页用 `navigateTo` 跳**：会失败，必须 `switchTab`。
2. **跳 tab 页时带 query**：`switchTab` 不支持参数，需要改用全局状态或存储。
3. **在组件里调用 `onPageScroll`**：只能在页面 setup 里注册。
4. **给 `page` 加 class 做主题**：小程序不支持，必须挂到页面根节点。
5. **用网络图做 CSS 背景**：小程序不显示，改用 `<image>`。
6. **直接操作 DOM / window / document**：小程序逻辑层没有 DOM。


## 第 10 章　自测题与动手练习

先自己回答，再看文末答案。

### 10.1 自测题

1. 新增一个 tab 页面，最少要改哪两个地方？
2. 详情页为什么用 `uni.navigateTo`，而不用 `uni.switchTab`？
3. `defineModel` 编译后相当于哪两件事？
4. 为什么 `useFavorites` 的 `ref` 要写在函数外面？
5. `onLoad` 和 `onMounted` 有什么区别？各适合做什么？
6. 主题切换为什么不能用 `<html data-theme>`？
7. 小程序里为什么不能 `document.querySelector`？替代思路是什么？
8. 图片要正式发布，`urlCheck: false` 足够吗？还需要做什么？
9. `switchTab` 能不能带 query？不能的话怎么传参？
10. 说出三个为 WXSS 兼容做的改动。

### 10.2 动手练习

1. 往 `destinations.js` 里加一个你喜欢的城市，让它出现在列表页；再把它设为 `featured: true`，看首页是否出现。
2. 把 `useFavorites` 的存储 key 改成 `my-favorites`，观察刷新后收藏是否还在，并解释原因。
3. 给首页功能卡加第 4 张，调整 `.grid-col` 的宽度让一行显示 4 张（提示：25%）。
4. 给详情页加一个「复制目的地名称」按钮，使用 `uni.setClipboardData`。
5. 把行程页的「随机灵感」改成「随机灵感 + 自动聚焦输入框」。
6. 把主题默认值改成深色，并确保刷新后仍然是深色。

### 10.3 参考答案

1. 新建页面文件，并在 `pages.json` 的 `pages` 和 `tabBar.list` 中各加一项（tabBar 最多 5 项）。
2. 详情页不在 `tabBar.list` 中；`switchTab` 只能跳 tab 页且不能带参数，`navigateTo` 可以带 query。
3. 一个 `modelValue` prop + 一个 `update:modelValue` 事件（命名 model 则是 `region` + `update:region`）。
4. 写在函数内每次调用都会创建新 `ref`，页面之间不同步；写在模块级才是单例。
5. `onLoad` 是 uni 页面生命周期，页面创建时执行一次，能接收路由 query；`onMounted` 是 Vue 组件挂载，适合初始化组件内部逻辑。
6. 小程序没有 `html` / `document` 节点；改用页面根节点 class + CSS 变量，原生导航栏用 `uni.setNavigationBarColor` 同步。
7. 小程序逻辑层与视图层分离，没有 DOM；用数据驱动视图，用 `uni.*` API 代替 DOM 操作。
8. 不够。`urlCheck:false` 只是开发者工具跳过校验；正式发布要在小程序后台配置 `downloadFile` 合法域名，或把图片放到 `src/static/` 用本地路径。
9. 不能。可以写到全局状态、本地存储，或先 `switchTab` 再由目标页从存储读取。
10. 例如：`color-mix()` → 固定 rgba；`aspect-ratio` → 固定高度；`<Transition>` → 移除；`<select>` → `<picker>`；网络背景图 → `<image>`。


## 附录　运行命令与排错

### 常用命令

```bash
cd D:/web/uniapp-app
npm install            # 首次安装依赖

npm run dev:h5         # H5 开发服务器
npm run dev:mp-weixin  # 监听编译到 dist/dev/mp-weixin
npm run dev:app        # 编译 App 资源到 dist/dev/app

npm run build:h5         # 生产构建 → dist/build/h5
npm run build:mp-weixin  # 生产构建 → dist/build/mp-weixin
npm run build:app        # 生产构建 → dist/build/app
```

### HBuilderX 运行

| 目标 | 菜单 |
| --- | --- |
| H5 | 运行 → 运行到浏览器 |
| 微信小程序 | 运行 → 运行到小程序模拟器 → 微信开发者工具 |
| Android | 运行 → 运行到手机或模拟器 → 运行到 Android App 基座 |

### 常见问题

| 现象 | 原因 | 解决 |
| --- | --- | --- |
| 微信开发者工具闪退 | 系统有 `ELECTRON_RUN_AS_NODE=1` | 启动前清除该环境变量 |
| CLI 提示 `IDE service port disabled` | 开发者工具未开服务端口 | 设置 → 安全设置 → 服务端口（手动开启） |
| 小程序图片不显示 | 域名未配置 / `urlCheck` | 开发期关闭校验；正式发布配置合法域名或本地化 |
| 跳 tab 页报错 | 用了 `navigateTo` | 改用 `switchTab` |
| 主题切换无效 | class 加在了 `page` 上 | 加到页面根节点 `.app` |
| 输入框聚焦无效 | 没有先置 false | `focused=false` 后延时 50ms 再设 true |
| GitHub Pages 404 | Pages 构建延迟 | 等待 1–2 分钟再刷新 |

### 在线预览

<https://bell-567.github.io/web/uniapp/>
